# A galactic-binary block for enchilada

This notebook plugs a **real LISA source-class sampler** into the enchilada
`Wheel`. To make it obvious *what is enchilada and what is not*, everything that
is **not** enchilada — the GBGPU waveform helpers, the LISA noise model, the data
injector, and the `GBBlock` itself — lives in a separate module,
[`gb_model.py`](gb_model.py). This notebook only:

1. imports enchilada's `L1Data` and `Wheel`,
2. imports the model pieces from `gb_model`,
3. builds the dataset, wraps it in an `L1Data`, runs it through a `Wheel`, and
   reads the result back.

So every enchilada touchpoint is visible here; the physics is behind the import.

- **waveforms** — [GBGPU](https://github.com/mikekatz04/GBGPU) narrowband TDI;
- **sampler** — [Eryn](https://github.com/mikekatz04/Eryn), inside the block;
- **noise** — fixed LISA sensitivity, converted to `DataCovariance`, held by Wheel, and consumed
  through the explicit `noise_covariance.noise_psd()` argument.

> **Requires** the LISA stack (`gbgpu`, `eryn`, `lisaanalysistools`) plus
> `matplotlib` and `corner` for the plots, in the same env as `enchilada` — the
> environment where these optional packages are installed. Outputs are not committed; run it to
> populate them.

The example requirements use NumPy `<2.4`: released Eryn 1.2.6 calls `numpy.in1d`, removed in NumPy 2.4. Install the requirements after syncing project extras and launch with `--no-sync` to preserve this environment. Dependency presence does not establish native GBGPU runtime validation.

In [ ]:
# Dependency preflight. This example needs an external LISA stack that is NOT an
# enchilada dependency -- see examples/requirements-gb.txt for the full matrix.
import importlib.util
import sys

# import name -> pip name (they differ for LISA Analysis Tools)
_NEEDED = {
    "gbgpu": "gbgpu",
    "eryn": "eryn",
    "lisatools": "lisaanalysistools",
    "matplotlib": "matplotlib",
    "corner": "corner",
}
_missing = sorted(
    pip for mod, pip in _NEEDED.items() if importlib.util.find_spec(mod) is None
)
if _missing:
    _py = f"{sys.version_info.major}.{sys.version_info.minor}"
    raise SystemExit(
        f"Missing: {', '.join(_missing)}\n\n"
        f"From the checkout root, in this order:\n"
        f"    uv sync --extra numeric-orbits --extra examples\n"
        f"    uv pip install -r examples/requirements-gb.txt\n"
        f"    uv run --no-sync jupyter lab examples/gb_block_eryn.ipynb\n\n"
        f"The requirements use NumPy <2.4 for released Eryn 1.2.6.\n"
        f"Supported: Python 3.12-3.13 on Linux or Apple-silicon macOS "
        f"(this is {_py} on {sys.platform}). gbgpu and lisaanalysistools ship "
        f"wheels but no sdist, so unsupported platforms report "
        f"'No matching distribution found' rather than a build error."
    )
print("dependency presence check: ok")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# enchilada — the orchestration layer
from enchilada import DataCovariance, L1Data, Wheel

# everything that is NOT enchilada (see gb_model.py)
from gb_model import FixedLISANoise, GBBlock, draw_gb_prior, inject_gb, waveform

## 1. Define the dataset

enchilada's frequency-domain convention is the one-sided rfft grid of the
underlying time series (`num_time_samples` counts *time-domain* samples; each channel array
has length `num_time_samples // 2 + 1`).

In [ ]:
dt = 15.0
N_time = 2**19  # time-domain samples -> pins Tobs, df
Tobs = N_time * dt  # ~91 days
df = 1.0 / Tobs
freqs = np.fft.rfftfreq(N_time, dt)
CHANS = ("A", "E")
NB = 128  # GBGPU template band width (bins)

TRUTH = np.array([2.0e-22, 3.0e-3, 1.0e-16, 0.7])  # amp, f0, fdot, phi0
ANGLES = (0.9, 1.3, 2.1, 0.4)  # iota, psi, lam, beta  (fixed)
PARAMS = ["amp", "f0", "fdot", "phi0"]
print(f"Tobs={Tobs / 86400:.0f} d, df={df:.2e} Hz, grid={freqs.size} bins")

## 2. Noise model + injected data (both from `gb_model`)

`FixedLISANoise` and `inject_gb` are user code. The injector returns channel arrays; wrapping them in `L1Data` establishes the observation grid. `DataCovariance.from_psd` converts the fixed one-sided PSD into coefficient covariance on that grid. Wheel stores it separately from the observations.


In [ ]:
noise = FixedLISANoise()
data, info = inject_gb(TRUTH, ANGLES, Tobs, dt, N_time, CHANS, noise, NB=NB, seed=42)
print(
    f"injected GB at f0={TRUTH[1]} Hz, bins {info['band']}, optimal SNR={info['snr']:.1f}"
)

# --- enchilada: the data contract ---
observed = L1Data(
    channel_data=data,
    sample_rate_hz=1.0 / dt,
    num_time_samples=N_time,
    channel_names=CHANS,
    tdi_generation="1.5",  # A1TDISens is the TDI 1.5 A channel
    physical_observable="fractional_frequency",
    data_domain="frequency",
)  # start_time_gps defaults to 0.0

## 3. Run the model through enchilada

The caller-side `draw_gb_prior` helper draws each walker from the declared uniform-box prior using a separate initialization RNG. The caller passes its complete result to `Wheel.add(initial_block_result=...)`; registration does not call the block or draw any parameters. Every `sample` call receives `conditional_residual`, `noise_covariance` and `current_block_result`. The block recreates its GBGPU and Eryn resources and restores numerical walker/RNG state from the `BlockResult`. It recomputes likelihoods for the conditional residual and uses Eryn's CPU stretch move with fixed ensemble splits, so all proposal randomness follows the saved Eryn stream.

The accepted signal is rendered from the first current walker, which represents a posterior draw after mixing. The callback collects each cycle's samples from the ledger; full chain history stays outside the block and the Wheel.

Registering the block with `data_domain="frequency"` asks Wheel to translate its inputs and accepted signal. The narrowband likelihood requires native frequency covariance with independent channels, as produced by `DataCovariance.from_psd`; a covariance translated from a different native domain generally has correlations between bins and cannot be used by this PSD likelihood. If observations and a native spectral covariance are stored in time or WDM, Wheel restores that native spectral covariance for the GB block.


In [ ]:
BOUNDS = [
    [0.3 * TRUTH[0], 3.0 * TRUTH[0]],
    [TRUTH[1] - 30 * df, TRUTH[1] + 30 * df],
    [0.0, 3.0e-16],
    [0.0, 2 * np.pi],
]

# MCMC sampling budget: total posterior samples = n_walkers * steps_per_cycle * n_cycles
n_walkers, steps_per_cycle, n_cycles = 40, 200, 12
block = GBBlock(
    bounds=BOUNDS,
    angles=ANGLES,
    band=NB,
    n_walkers=n_walkers,
    steps_per_cycle=steps_per_cycle,
)

noise_covariance = DataCovariance.from_psd(
    reference_data=observed, noise_psd_model=noise
)
# Distinct child streams make initialization and sampling reproducible
# without replaying the same draws in both stages.
initialization_seed, sampling_seed = np.random.SeedSequence(0).spawn(2)
initialization_rng = np.random.default_rng(initialization_seed)
wheel = Wheel(
    observed,
    initial_noise_covariance=noise_covariance,
    random_seed=int(sampling_seed.generate_state(1)[0]),
)  # enchilada
initial = draw_gb_prior(block, observed, noise_covariance, rng=initialization_rng)
wheel.add(
    block, initial_block_result=initial, data_domain="frequency"
)  # enchilada delivers the GB input grid


chain_chunks = []


def report(it, w):
    current_block_result = w.ledger["gb"]
    chain_chunks.append(current_block_result.sampler_state["samples"])
    p = [current_block_result.model_parameters[name] for name in PARAMS]
    print(
        f"  cycle {it + 1:2d}: amp={p[0]:.3e}  f0={p[1]:.9f}  fdot={p[2]:.3e}  phi0={p[3]:.3f}"
    )


total = n_walkers * steps_per_cycle * n_cycles
print(
    f"running the Wheel: {total:,} GB samples "
    f"({n_walkers} walkers x {steps_per_cycle} steps x {n_cycles} cycles), "
    f"noise via explicit DataCovariance ..."
)
wheel.run(
    n_cycles, on_cycle_complete=report
)  # enchilada drives block.sample each cycle

## 4. Inspect recovery and mixing

Initialization now covers the full uniform prior, so burn-in and mixing must be assessed for each run. A single-temperature ensemble can remain in secondary frequency modes, especially from a wide prior. Increase the sampling budget or extend the block with parallel tempering (including its evolving temperature state) before treating an unconverged run as a posterior result.

The cells below first report the full post-burn-in chain and its frequency occupancy. An optional summary conditioned on the injection's frequency bin is useful for this synthetic diagnostic, but it is not the full posterior and the injected truth must not be used to select modes in a real analysis. `fdot` is weakly constrained over this short observation; compare its marginal with the prior.


In [ ]:
chain = np.concatenate(chain_chunks)
post_all = chain[len(chain) // 2 :]  # provisional first-half burn-in

print("full post-burn-in ensemble: mean +/- std vs truth")
for i, p in enumerate(PARAMS):
    print(
        f"  {p:5s}: {post_all[:, i].mean():.6e} +/- {post_all[:, i].std():.2e}"
        f"   truth {TRUTH[i]:.6e}"
    )

in_injected_bin = np.abs(post_all[:, 1] - TRUTH[1]) < 0.5 * df
print(f"samples within 0.5 df of injection: {in_injected_bin.mean():.1%}")
# Plot every mode. Do not quietly discard walkers based on the known injection.
post = post_all

In [ ]:
import corner

SCALE = np.array([1e-22, 1e-9, 1e-16, 1.0])
OFFSET = np.array([0.0, 3.0e-3, 0.0, 0.0])
LABELS = [
    r"$\mathcal{A}\,/\,10^{-22}$",
    r"$(f_0 - 3\,\mathrm{mHz})\,/\,\mathrm{nHz}$",
    r"$\dot{f}\,/\,10^{-16}\,\mathrm{Hz\,s^{-1}}$",
    r"$\phi_0$  [rad]",
]

fig = corner.corner(
    (post - OFFSET) / SCALE,
    labels=LABELS,
    truths=(TRUTH - OFFSET) / SCALE,
    truth_color="#c1121f",
    levels=(1 - np.exp(-0.5), 1 - np.exp(-2.0), 1 - np.exp(-4.5)),
    bins=40,
    max_n_ticks=4,
    labelpad=0.09,
    smooth=0.9,
    smooth1d=0.9,
    quantiles=[0.16, 0.5, 0.84],
    show_titles=True,
    title_fmt=".2f",
    fill_contours=True,
    plot_datapoints=False,
    plot_density=False,
    color="#1d3557",
    hist_kwargs={"lw": 1.3},
    contour_kwargs={"linewidths": 0.7},
    label_kwargs={"fontsize": 10},
    title_kwargs={"fontsize": 9},
)
fig.set_size_inches(7.6, 7.6)
for ax in fig.get_axes():
    ax.tick_params(axis="both", labelsize=8, pad=2)
    for lbl in (*ax.get_xticklabels(), *ax.get_yticklabels()):
        lbl.set_rotation(0)
        lbl.set_ha("center" if lbl in ax.get_xticklabels() else "right")
fig.suptitle(
    f"Galactic binary ensemble through the Wheel  |  SNR {info['snr']:.0f}  |  "
    f"{n_cycles} cycles, {len(post):,} samples  |  red = injected truth",
    fontsize=10.5,
    y=1.04,
)
plt.show()

In [ ]:
band = info["band"]
rsi, rhA, rhE = waveform(post.mean(0), ANGLES, Tobs, dt, NB)
fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(
    freqs[band] * 1e3,
    np.abs(observed.channel_data["A"][band]),
    color="0.7",
    label="data |A|",
)
ax.semilogy(
    freqs[band] * 1e3,
    np.abs(info["signal"]["A"][band]),
    "C0",
    lw=2,
    label="injected |A|",
)
ax.semilogy(
    freqs[rsi : rsi + NB] * 1e3, np.abs(rhA), "C3--", lw=2, label="recovered |A|"
)
ax.set_xlabel("frequency [mHz]")
ax.set_ylabel("|A(f)|")
ax.legend()
ax.set_title("Galactic binary: data, injection, recovered model")
plt.tight_layout()
plt.show()

## Where next

- **All eight parameters** — extend `BOUNDS` and the block's parameter mapping; the four sky/orientation angles are fixed in this example.
- **A handful of sources** — register several configured `GBBlock`s with unique `name=` values. Wheel hands each source the data minus every other source's accepted signal contribution and returns its own `current_block_result` separately.
- **Sampled noise** — register a noise block that publishes `DataCovariance` through `BlockResult(noise_covariance=...)`, with no signal arrays. GBBlock reads the explicit covariance on every `sample` call and recomputes all walker likelihoods.
- **Check mixing** — compare independent campaigns, inspect full-chain frequency modes and traces, and carry any adaptive/tempering state explicitly when extending the Eryn wrapper.
